# Red Bus Data Analysis 2

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow

In [2]:
#Mounting drive to get my data
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
#Load path where my dataset is
path = "/content/drive/MyDrive/RedBusAnalysis"

In [4]:
#Importing pandas
import pandas as pd

In [5]:
#Loading data set
train_data = pd.read_csv(path + "/train.csv")
transaction_data = pd.read_csv(path + "/transactions.csv")
test_data = pd.read_csv(path + "/test_8gqdJqH.csv")

# Feature Extraction

In [6]:
#Filtering for prediction 15 days before journey
transaction_15 = transaction_data[transaction_data["dbd"] == 15]

In [7]:
#Creating unique route key to match later with test dataset
transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)

<ipython-input-7-1703587420>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)


In [8]:
#Selecting Relevant feature
features = transaction_15[["route_key", "cumsum_seatcount", "cumsum_searchcount", "srcid_region", "destid_region", "srcid_tier", "destid_tier"]]

In [10]:
#Merge with train labels
train_data["route_key"] = train_data["doj"] + "_" + train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
#Drop if existing feature in train data to avoid collision during merge
cols_to_drop = ["cumsum_seatcount", "cumsum_searchcount",
                "srcid_region", "destid_region", "srcid_tier", "destid_tier"]

train_data = train_data.drop(columns=[col for col in cols_to_drop if col in train_data.columns])

train_data = train_data.merge(features, on="route_key", how="left")
train_data.dropna(inplace=True)
train_data.head()

,doj,srcid,destid,final_seatcount,route_key,cumsum_seatcount,cumsum_searchcount,srcid_region,destid_region,srcid_tier,destid_tier
0,2023-03-01,45,46,2838.0,2023-03-01_45_46,16.0,480.0,Karnataka,Tamil Nadu,Tier 1,Tier 1
1,2023-03-01,46,45,2298.0,2023-03-01_46_45,34.0,352.0,Tamil Nadu,Karnataka,Tier 1,Tier 1
2,2023-03-01,45,47,2720.0,2023-03-01_45_47,36.0,892.0,Karnataka,Andhra Pradesh,Tier 1,Tier 1
3,2023-03-01,47,45,2580.0,2023-03-01_47_45,18.0,1130.0,Andhra Pradesh,Karnataka,Tier 1,Tier 1
4,2023-03-01,46,9,4185.0,2023-03-01_46_9,48.0,1023.0,Tamil Nadu,Tamil Nadu,Tier 1,Tier2


In [11]:
#Mergin with test set
duplicate_cols = [
    "cumsum_seatcount", "cumsum_searchcount",
    "srcid_region", "destid_region",
    "srcid_tier", "destid_tier"
]

# Drop them from test_data if they exist
test_data = test_data.drop(columns=[col for col in duplicate_cols if col in test_data.columns])
test_data = test_data.merge(features, on="route_key", how="left")
test_data['cumsum_seatcount'] = test_data['cumsum_seatcount'].fillna(0)
test_data['cumsum_searchcount'] = test_data['cumsum_searchcount'].fillna(0)
for col in ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]:
    test_data[col] = test_data[col].fillna("Unknown")
test_data.head()

,route_key,doj,srcid,destid,cumsum_seatcount,cumsum_searchcount,srcid_region,destid_region,srcid_tier,destid_tier
0,2025-02-11_46_45,2025-02-11,46,45,38.0,1082.0,Tamil Nadu,Karnataka,Tier 1,Tier 1
1,2025-01-20_17_23,2025-01-20,17,23,0.0,1175.0,East 1,East 1,Tier2,Tier 1
2,2025-01-08_02_14,2025-01-08,2,14,0.0,0.0,Unknown,Unknown,Unknown,Unknown
3,2025-01-08_08_47,2025-01-08,8,47,0.0,0.0,Unknown,Unknown,Unknown,Unknown
4,2025-01-08_09_46,2025-01-08,9,46,0.0,0.0,Unknown,Unknown,Unknown,Unknown


In [13]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [14]:
# Encoding categorical features safely
categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]
for col in categorical_features:
    le = LabelEncoder()

    # Combine train + test categories for fitting
    combined_values = pd.concat([train_data[col], test_data[col]], axis=0).astype(str)

    # Fit encoder on all possible values
    le.fit(combined_values)

    # Transform separately
    train_data[col] = le.transform(train_data[col].astype(str))
    test_data[col] = le.transform(test_data[col].astype(str))

In [15]:
#Select final features
features = ["cumsum_seatcount", "cumsum_searchcount"]
target = "final_seatcount"

In [16]:
X = train_data[features].values
Y = train_data[target].values
X_test = test_data[features].values

In [17]:
#Normalizing features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

In [18]:
!pip install xgboost lightgbm catboost optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 15.3 MB/s eta 0:00:00


In [19]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna

In [20]:
#Split data
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

In [26]:
# Model 1: XGBoost + GridSearchCV
print("Training XGBoost with GridSearchCV")
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
}

grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='neg_root_mean_squared_error', verbose=1)
grid_xgb.fit(X_train, y_train)

Training XGBoost with GridSearchCV
Fitting 3 folds for each of 8 candidates, totalling 24 fits


GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, gamma=None,
                                    grow_policy=None, importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, m...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None,
                                    random_state=42, ...),
             param_grid={'learning_rate': [0.01, 0.1], 'max_depth': [3, 5],
                         'n_estimators': [100, 200]},
             scoring='neg_root_mean_squared_error', verbose=1)

In [27]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_val, grid_xgb.predict(X_val))
rmse = np.sqrt(mse)
print("Best XGBoost params:", grid_xgb.best_params_)
print("XGBoost RMSE:", rmse)

Best XGBoost params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
XGBoost RMSE: 843.5777120402662


In [31]:
test_prediction = grid_xgb.predict(X_test)

In [32]:
test_prediction

array([3359.4778 , 1836.4945 ,  523.02124, ..., 1600.1849 ,  523.02124,
       1552.8015 ], dtype=float32)

In [34]:
submission = test_data[["route_key"]].copy()
submission["final_seatcount"] = test_prediction

In [35]:
submission.to_csv("submission_file.csv", index=False)

In [36]:
#Download submission file
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
submission

,route_key,final_seatcount
0,2025-02-11_46_45,3359.477783
1,2025-01-20_17_23,1836.494507
2,2025-01-08_02_14,523.021240
3,2025-01-08_08_47,523.021240
4,2025-01-08_09_46,523.021240
...,...,...
5895,2025-01-23_46_48,3491.418701
5896,2025-02-21_46_09,523.021240
5897,2025-01-17_32_19,1600.184937
5898,2025-01-24_45_03,523.021240
